# Building Graph Workflows with the Agent Development Kit (ADK 2.x)

## Overview

In complex agentic applications, linear agent chains or simple sub-agent routing are often not flexible enough. Google Agent Development Kit (ADK) 2.x introduces **Graph Workflows** (`Workflow`), enabling non-linear execution flows including conditional routing, parallel processing, iterative refinement loops, and human-in-the-loop interventions.

In this notebook, you will learn how to build four common graph workflow patterns using ADK 2.x:

1. **Parallel Processing Workflow (`JoinNode`)**: Run multiple processing nodes in parallel and aggregate their results using a synchronization node.
2. **Evaluator-Optimizer Refinement Loop**: Build an iterative loop where an evaluator agent grades generated content and loops back with feedback until quality criteria are met.
3. **Routing & Branching Workflow**: Classify incoming requests with an LLM agent and route them dynamically to specialized handler nodes (supporting single or multi-branch parallel execution).
4. **Human-in-the-Loop (HITL) Workflow**: Suspend workflow execution using `RequestInput` to solicit human review/feedback before resuming or branching.

Each workflow will be written to disk using `%%writefile` and tested interactively using the **ADK Developer UI** (`adk web`).

## Environment Setup & Configuration

First, let's configure the environment variables required by Google Cloud Vertex AI and ADK.

In [ ]:
import os

LOCATION = "global"
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"

In [ ]:
%%bash
echo > adk_workflows/.env "GOOGLE_CLOUD_LOCATION=$GOOGLE_CLOUD_LOCATION
GOOGLE_GENAI_USE_VERTEXAI=$GOOGLE_GENAI_USE_VERTEXAI"


---
## 1. Parallel Processing Workflow (`JoinNode`)

Parallel workflows allow executing multiple independent tasks concurrently and waiting for all branches to finish before passing their combined state to a downstream aggregation node.

### Workflow Architecture
- **Parallel Start Nodes (`node_A`, `node_B`, `node_C`)**: Python nodes executed concurrently when triggered from `START`.
- **Synchronization Node (`JoinNode`)**: Waits for all incoming parallel edges to complete.
- **Aggregation Node (`node_D`)**: Receives the collected state keys (`branch_A`, `branch_B`, `branch_C`) and formats the final summary.

In [ ]:
!mkdir -p ./adk_workflows/graph1_parallel

In [ ]:
%%writefile ./adk_workflows/graph1_parallel/agent.py
"""ADK 2.x Parallel Processing Workflow."""

# pylint: disable=invalid-name

from google.adk import Event, Workflow
from google.adk.workflow import JoinNode


# 1. Define Parallel Processing Nodes (A, B, C)
def node_A(node_input: str):
    return Event(
        message=f"Node A executed... input={node_input}",
        output=int(node_input),
    )


def node_B(node_input: str):
    input_b = int(node_input)
    return Event(
        message=f"Node B executed... input={node_input}",
        output=input_b * 100,
    )


def node_C(node_input: str):
    input_c = int(node_input)
    return Event(
        message=f"Node C executed... input={node_input}",
        output=input_c * input_c,
    )


# 2. Define Aggregation and Display Node (D)
def node_D(node_input: dict) -> Event:
    """Collects outputs from JoinNode, calculates the sum, and displays it."""
    val_a = node_input.get("node_A", 0.0)
    val_b = node_input.get("node_B", 0.0)
    val_c = node_input.get("node_C", 0.0)

    total_sum = val_a + val_b + val_c

    display_message = (
        f"### Execution Complete!\n\n"
        f"Successfully collected parallel outputs:\n"
        f"- **Node A Output:** `{val_a}`\n"
        f"- **Node B Output:** `{val_b}`\n"
        f"- **Node C Output:** `{val_c}`\n\n"
        f"---\n"
        f"### Result\n"
        f"**Node D (Total Sum):** `{total_sum}`"
    )

    return Event(message=display_message, output=total_sum)


join_node = JoinNode(name="join_node")

# 3. Construct Workflow
root_agent = Workflow(
    name="parallel_workflow",
    edges=[
        ("START", node_A, join_node),
        ("START", node_B, join_node),
        ("START", node_C, join_node),
        (join_node, node_D),
    ],
)


### Open the ADK Developer UI

You can launch the ADK Dev UI to interact with your agents using the `adk web` command. 

Execute the command below and click on the generated web link to open the interface.

In [ ]:
# On Cloud Workstations
!adk web adk_workflows --allow_origins "regex:https://.*\.cloudworkstations\.dev"

**Note:** if you are using Agent Platform Workbench, remove the comment out and run the cell below.

In [ ]:
# %%bash
# PROXY_BASE=$(curl -s http://metadata.google.internal/computeMetadata/v1/instance/attributes/proxy-url -H "Metadata-Flavor: Google")
# ORIGIN="${PROXY_BASE%/}"
# adk web adk_workflows --allow_origins "$ORIGIN"

### Use the Developer UI
Select `graph1_parallel` on the top left, and send queries referring to the instructions below:
- Enter an initial value, e.g. `"5"`.
- Observe how Nodes A, B, and C execute in parallel, and `JoinNode` gathers their state variables for Node D to aggregate.

---
## 2. Evaluator-Optimizer Refinement Loop

Iterative refinement loops evaluate the output of a generator against defined constraints or quality metrics, feeding feedback back into the generator until criteria are met.

### Workflow Architecture
- **Input Processor (`process_input`)**: Saves the user's initial topic to shared workflow state (`{topic}`).
- **Generator Agent (`generate_headline`)**: Writes a headline incorporating state variables `{topic}` and optional `{feedback?}`.
- **Evaluator Agent (`evaluate_headline`)**: Uses a structured Pydantic schema (`Feedback`) to grade whether the headline is `"tech-related"` or `"unrelated"`.
- **Router Node (`route_headline`)**: Emits `Event(route=feedback.grade)`. If `"unrelated"`, the edge loops back to `generate_headline`.

In [ ]:
!mkdir -p ./adk_workflows/graph2_loop

In [ ]:
%%writefile ./adk_workflows/graph2_loop/agent.py
"""ADK 2.x Evaluator-Optimizer Refinement Loop Workflow."""

from typing import Literal

from google.adk import Agent, Event, Workflow
from pydantic import BaseModel, Field

MODEL = "gemini-2.5-flash"


class Feedback(BaseModel):
    grade: Literal["tech-related", "unrelated"] = Field(
        description=(
            "Decide if the headline is related to technology or software "
            "engineering."
        ),
    )
    feedback: str = Field(
        description=(
            "If the headline is unrelated to technology, provide feedback "
            "on how to make it more tech-focused."
        ),
    )


def process_input(node_input: str):
    """Puts user input in the state."""
    return Event(state={"topic": node_input})


generate_headline = Agent(
    name="generate_headline",
    model=MODEL,
    instruction="""
    Write a headline about the topic "{topic}".
    If feedback is provided, take it into account.
    The feedback: {feedback?}
    """,
)

evaluate_headline = Agent(
    name="evaluate_headline",
    model=MODEL,
    instruction="""
    Grade whether the headline is related to technology or software engineering.
    """,
    output_schema=Feedback,
    output_key="feedback",
)


def route_headline(node_input: Feedback):
    return Event(route=node_input.grade)


root_agent = Workflow(
    name="evaluator_optimizer_loop",
    edges=[
        (
            "START",
            process_input,
            generate_headline,
            evaluate_headline,
            route_headline,
        ),
        # The Refinement Loop:
        (route_headline, {"unrelated": generate_headline}),
    ],
)


### Open the ADK Developer UI

You can launch the ADK Dev UI to interact with your agents using the `adk web` command. 

Execute the command below and click on the generated web link to open the interface.

In [ ]:
# On Cloud Workstations
!adk web adk_workflows --allow_origins "regex:https://.*\.cloudworkstations\.dev"

**Note:** if you are using Agent Platform Workbench, remove the comment out and run the cell below.

In [ ]:
# %%bash
# PROXY_BASE=$(curl -s http://metadata.google.internal/computeMetadata/v1/instance/attributes/proxy-url -H "Metadata-Flavor: Google")
# ORIGIN="${PROXY_BASE%/}"
# adk web adk_workflows --allow_origins "$ORIGIN"

### Use the Developer UI
Select `graph2_loop` on the top left, and send queries referring to the instructions below:
- Enter a non-tech topic, e.g.: `"Baking a delicious chocolate cake"`
- Observe the loop in action as the evaluator rejects the non-tech headline, sends feedback, and the generator refines it until it produces a tech-focused headline.

---
## 3. Routing & Branching Workflow

In a routing workflow, an incoming request is classified into one or more categories, and execution is dynamically routed to the appropriate destination node(s). If multiple categories are identified, ADK automatically executes the target branches in parallel.

### Workflow Architecture
- **Classifier Node (`process_message`)**: An LLM Agent (`gemini-3.5-flash`) that analyzes user input and outputs categories (`BUG`, `CUSTOMER_SUPPORT`, `LOGISTICS`).
- **Router Node (`router`)**: A Python function that splits the LLM output into routes and emits `Event(route=...)`.
- **Destination Nodes**: Specific python handler functions (`response_1_bug`, `response_2_support`, `response_3_logistics`).

In [ ]:
!mkdir -p ./adk_workflows/graph3_router

In [ ]:
%%writefile ./adk_workflows/graph3_router/agent.py
"""ADK 2.x Routing Workflow."""

from google.adk import Agent, Event, Workflow

MODEL = "gemini-2.5-flash"

# 1. LLM Classifier Agent
process_message = Agent(
    name="process_message",
    model=MODEL,
    instruction="""Classify user message into either "BUG", "CUSTOMER_SUPPORT",
     or "LOGISTICS". If you think a message applies to more than one category,
     reply with a comma separated list of categories.
  """,
    output_schema=str,
)


# 2. Router Node
def router(node_input: str):
    # Split the comma-separated string from the LLM
    routes = node_input.split(",")
    # Clean up any whitespace
    routes = [route.strip() for route in routes]

    # Emitting an Event with 'route' dictates the next edge(s) to traverse
    return Event(route=routes)


# 3. Destination Nodes
def response_1_bug():
    return Event(message="Handling bug...")


def response_2_support():
    return Event(message="Handling customer support...")


def response_3_logistics():
    return Event(message="Handling logistics...")


# 4. Construct Workflow
root_agent = Workflow(
    name="routing_workflow",
    edges=[
        # 1. Start execution at the LLM classifier, then pass output to router
        ("START", process_message, router),
        # 2. Map the routes emitted by the router to the destination nodes
        (
            router,
            {
                "BUG": response_1_bug,
                "CUSTOMER_SUPPORT": response_2_support,
                "LOGISTICS": response_3_logistics,
            },
        ),
    ],
)


### Open the ADK Developer UI

You can launch the ADK Dev UI to interact with your agents using the `adk web` command. 

Execute the command below and click on the generated web link to open the interface.

In [ ]:
# On Cloud Workstations
!adk web adk_workflows --allow_origins "regex:https://.*\.cloudworkstations\.dev"

**Note:** if you are using Agent Platform Workbench, remove the comment out and run the cell below.

In [ ]:
# %%bash
# PROXY_BASE=$(curl -s http://metadata.google.internal/computeMetadata/v1/instance/attributes/proxy-url -H "Metadata-Flavor: Google")
# ORIGIN="${PROXY_BASE%/}"
# adk web adk_workflows --allow_origins "$ORIGIN"

### Use the Developer UI
Select `graph3_router` on the top left, and send queries referring to the instructions below:
- Try sending a query that crosses multiple categories, e.g.: `"I can't track my shipment, and the mobile app keeps crashing!"`
- Notice how both the **BUG** and **LOGISTICS** destination nodes execute.

---
## 4. Human-in-the-Loop (HITL) Workflow

Workflows often require human approval or intervention before taking irreversible actions. In ADK, yielding a `RequestInput` object pauses workflow execution until the user provides input or feedback.

### Workflow Architecture
- **Input Processor (`process_input`)**: Initializes workflow state with customer complaint and empty feedback.
- **Draft Email Agent (`draft_email`)**: LLM Agent drafting a customer service response based on `{complaint}` and `{feedback?}`.
- **Request Review Node (`request_human_review`)**: Yields `RequestInput`, pausing execution and prompting the human reviewer.
- **Handle Review Router (`handle_human_review`)**: Evaluates human response: `"approve"` -> `send_email`, `"reject"` -> `reject_email`, or revision feedback -> loops back to `draft_email`.

In [ ]:
!mkdir -p ./adk_workflows/graph4_hitl

In [ ]:
%%writefile ./adk_workflows/graph4_hitl/agent.py
"""ADK 2.x Human-in-the-Loop Workflow."""

from google.adk import Agent, Event, Workflow
from google.adk.events import RequestInput

MODEL = "gemini-2.5-flash"


def process_input(node_input: str):
    """Takes initial customer complaint as input and sets state."""
    return Event(state={"complaint": node_input, "feedback": ""})


draft_email = Agent(
    name="draft_email",
    model=MODEL,
    instruction="""
    Please write a polite, helpful response email to customer complaint:
    "{complaint}"
    If there is feedback from the manager to revise draft, incorporate it:
    "{feedback?}"
    """,
    output_key="draft",
)


def request_human_review(draft: str):
    return RequestInput(
        message=(
            "Please review the following draft email and provide "
            f"'approve', 'reject', or feedback to revise.\n\n---\n{draft}\n---"
        )
    )


def handle_human_review(node_input: str):
    user_resp = node_input.lower().strip()
    if user_resp == "reject":
        return Event(route="rejected")
    if user_resp == "approve":
        return Event(route="approved")
    return Event(state={"feedback": node_input}, route="revise")


def reject_email():
    return Event(message="Draft rejected.")


def send_email(draft: str):
    # pylint: disable=unused-argument
    return Event(message="Draft approved and sent successfully.")


root_agent = Workflow(
    name="hitl_workflow",
    edges=[
        (
            "START",
            process_input,
            draft_email,
            request_human_review,
            handle_human_review,
        ),
        (
            handle_human_review,
            {
                "revise": draft_email,
                "approved": send_email,
                "rejected": reject_email,
            },
        ),
    ],
)


### Open the ADK Developer UI

You can launch the ADK Dev UI to interact with your agents using the `adk web` command. 

Execute the command below and click on the generated web link to open the interface.

In [ ]:
# On Cloud Workstations
!adk web adk_workflows --allow_origins "regex:https://.*\.cloudworkstations\.dev"

**Note:** if you are using Agent Platform Workbench, remove the comment out and run the cell below.

In [ ]:
# %%bash
# PROXY_BASE=$(curl -s http://metadata.google.internal/computeMetadata/v1/instance/attributes/proxy-url -H "Metadata-Flavor: Google")
# ORIGIN="${PROXY_BASE%/}"
# adk web adk_workflows --allow_origins "$ORIGIN"

### Use the Developer UI
Select `graph4_hitl` on the top left, and send queries referring to the instructions below:
1. Send an initial complaint, e.g.: `"My delivery was 3 days late and the box was crushed!"`
2. Observe how the workflow pauses execution and displays a prompt for human review.
3. Type revision feedback in the UI, e.g.: `"Too formal, make it shorter and offer a 10% discount."` to see the workflow update the draft.
4. Reply `"approve"` or `"reject"` to finish the workflow.

Copyright 2026 Google LLC

Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.